In [17]:
import sys

!{sys.executable} -m pip install neo4j pandas


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip3 install --upgrade pip


In [1]:
from neo4j import GraphDatabase
import pandas as pd

print("neo4j y pandas importados correctamente")

neo4j y pandas importados correctamente


In [2]:
from neo4j import GraphDatabase
import pandas as pd
from IPython.display import display, Markdown

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

print("Conexión correcta con Neo4j")

Conexión correcta con Neo4j


In [3]:
def ejecutar_query(titulo, query, params=None):
    display(Markdown(f"## {titulo}"))
    
    with driver.session(database="neo4j") as session:
        result = session.run(query, params or {})
        data = [record.data() for record in result]
    
    df = pd.DataFrame(data)
    
    if df.empty:
        print("La consulta no devolvió resultados.")
    else:
        display(df)
    
    return df

In [4]:
query1 = """
MATCH (i:Item)
WHERE NOT EXISTS { (:Recipe)-[:PRODUCES]->(i) }
  AND NOT EXISTS { (:Monster)-[:DROPS]->(i) }
  AND (
    EXISTS { (:WeaponCraft)-[:NEEDS]->(i) }
    OR EXISTS { (:WeaponUpgrade)-[:NEEDS]->(i) }
  )
RETURN i.name AS nombre, i.description AS descripcion
"""

df1 = ejecutar_query(
    "1. Objetos que no se crean ni se obtienen de monstruos, pero se usan en crafteo/mejora",
    query1
)

## 1. Objetos que no se crean ni se obtienen de monstruos, pero se usan en crafteo/mejora

,nombre,descripcion
0,Carbalita,Mineral obtenido de afloramientos. Aún se está...
1,Cristal de tierra,Microbios cristalizados obtenidos de afloramie...
2,Hueso misterioso,Hueso erosionado tomado de una pila de huesos....
3,Hueso colosal,Hueso gigantesco de una bestia. Se ha cortado ...
4,Reliquia de dracohueso,Hueso de gran calidad. Son los restos de un dr...
5,Caparazón de Bulaqchi,Caparazón de Bulaqchi. Ideal para artesanías g...
6,Colmillo agudo,"Hay colmillos de todo tipo. Este es pequeño, p..."
7,Esquirla de cólera,Fragmento de un cristal perturbador que posee ...
8,Coraza de Bulaqchi,Coraza de Bulaqchi de gran calidad. Tan fácil ...
9,Caparazón de Comaqchi,Caparazón de Comaqchi. Posee un tacto suave y ...


En esta consulta buscamos materiales que son un poco “externos” al sistema de obtención normal: no salen como resultado de una receta y tampoco los dropea ningún monstruo, pero sí aparecen como materiales necesarios para crear o mejorar armas. Lo importante aquí es usar `NOT EXISTS` para descartar los objetos producibles por `Recipe` y los que se obtienen por `Monster`, y después comprobar que aun así se usan en algún `WeaponCraft` o `WeaponUpgrade`.

También es importante haber puesto el `OR` entre paréntesis. Si no se hace, Neo4j podría interpretar mal la prioridad lógica entre `AND` y `OR`. El resultado devuelve **29 items**, como `Carbalita`, `Cristal de tierra`, `Hueso misterioso`, `Piedra de fuego`, `Fucium`, `Hierro`, etc. Esto encaja bastante bien con la idea de materiales base, minerales, huesos o vales que el modelo no representa como fabricables ni como drops, pero que sí son necesarios en procesos de crafteo o mejora.


### Consulta 2:
El id de receta, nombre de los items usados como input y nombre del item generado
como output de las 5 recetas que más ganancias generan. La ganancia que genera
una receta es el valor total de los items que genera, menos el coste total de los items
que necesita. Todos los items que se usan en la receta no deben ser producidos por
otras recetas.

In [5]:
query2 = """
MATCH (r:Recipe)-[p:PRODUCES]->(output:Item)
MATCH (r)-[:NEEDS]->(input:Item)

WITH r,
     collect(input) AS inputs,
     output,
     p.amount AS outAmount

WHERE ALL(i IN inputs WHERE NOT EXISTS { (:Recipe)-[:PRODUCES]->(i) })

WITH r,
     output,
     outAmount,
     inputs,
     reduce(
       coste = 0,
       i IN inputs | coste + coalesce(i.value, 0)
     ) AS costeTotal

WITH r,
     [i IN inputs | i.name] AS itemsInput,
     output.name AS itemOutput,
     coalesce(output.value, 0) * coalesce(outAmount, 1) - costeTotal AS ganancia

RETURN r.id AS recetaId,
       itemsInput,
       itemOutput,
       ganancia
ORDER BY ganancia DESC
LIMIT 5
"""

df2 = ejecutar_query(
    "2. Top 5 recetas con más ganancia",
    query2
)

## 2. Top 5 recetas con más ganancia

,recetaId,itemsInput,itemOutput,ganancia
0,244,"[Brote nocturno, Esencia de bicho divino]",Fuente de vida,530
1,243,"[Esencia de bicho divino, Semilla armadura]",Polvo de cáscara,85
2,204,"[Seta azul, Esencia de bicho divino]",Polvo de vida,68
3,209,"[Condensador electrobicho, Herramientas para t...",Trampa eléctrica,60
4,242,"[Esencia de bicho divino, Semilla de poder]",Polvo de demonio,55


Aquí calculamos las 5 recetas que más beneficio generan. Primero cogemos cada receta, su item de salida y todos sus inputs. La parte clave es que usamos `collect(input)` antes del filtro con `ALL`. Esto lo hicimos así porque si filtrásemos input por input, una receta podría colarse aunque solo uno de sus materiales cumpliese la condición. Con `ALL`, en cambio, obligamos a que **todos** los inputs de la receta no sean producidos por otras recetas.

Después usamos `reduce` para sumar el coste de los inputs y calculamos la ganancia como `valor del output * cantidad producida - coste total de inputs`. También se usa `coalesce` para evitar problemas si algún valor viene como `null`.

El resultado muestra que la receta más rentable es la **244**, que produce `Fuente de vida` y tiene una ganancia de **530**, bastante por encima del resto. Luego aparecen `Polvo de cáscara`, `Polvo de vida`, `Trampa eléctrica` y `Polvo de demonio`, con ganancias menores. Esto tiene sentido porque son objetos consumibles o útiles que se generan a partir de pocos materiales.


### Consulta 3:
El nombre y el tipo del arma que se puede craftear directamente y que más daño
hace de tipo fuego. Además, incluir los monstruos que se deben matar para crearla,
así como cualquier material extra que se necesario para fabricar el arma y que no se
obtiene matando dichos monstruos.

In [6]:
query3 = """
MATCH (:Element {name:'fire'})<-[d:DEALS]-(w:Weapon)<-[:PRODUCES]-(wc:WeaponCraft)

WITH w, wc, d
ORDER BY d.damage DESC
LIMIT 1

MATCH (wc)-[:NEEDS]->(i:Item)
OPTIONAL MATCH (m:Monster)-[:DROPS]->(i)

WITH w,
     collect(DISTINCT m.name) AS monstersRaw,
     collect(DISTINCT CASE WHEN m IS NULL THEN i.name END) AS extrasRaw

RETURN w.name AS nombre,
       w.kind AS tipo,
       [m IN monstersRaw WHERE m IS NOT NULL] AS monstruos,
       [e IN extrasRaw WHERE e IS NOT NULL] AS materialesExtra
"""

df3 = ejecutar_query(
    "3. Arma crafteable directamente con más daño de fuego",
    query3
)

## 3. Arma crafteable directamente con más daño de fuego

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (:Element {name:'fire'})<-[d:DEALS]-(w:Weapon)<-[:PRODUCES]-(wc:WeaponCraft)\n\nWITH w, wc, d\nORDER BY d.damage DESC\nLIMIT 1\n\nMATCH (wc)-[:NEEDS]->(i:Item)\nOPTIONAL MATCH (m:Monster)-[:DROPS]->(i)\n\nWITH w,\n     collect(DISTINCT m.name) AS monstersRaw,\n     collect(DISTINCT CASE WHEN m IS NULL THEN i.name END) AS extrasRaw\n\nRETURN w.name AS nombre,\n       w.kind AS tipo,\n       [m IN monstersRaw WHERE m IS NOT NULL] AS monstruos,\n       [e IN extrasRaw WHERE e IS NOT NULL] AS m

,nombre,tipo,monstruos,materialesExtra
0,Decapitagallinas I,great-sword,[Yian Kut-Ku],[Carbalita]


En esta consulta primero elegimos el arma que se puede crear directamente y que más daño de fuego hace. Por eso empezamos desde `Element {name: 'fire'}` y seguimos la relación `DEALS` hasta `Weapon`, asegurándonos además de que esa arma viene de un `WeaponCraft`, es decir, que es crafteable directamente. Después ordenamos por `d.damage DESC` y usamos `LIMIT 1` para quedarnos solo con la mejor.

Una vez elegida el arma, miramos los items que necesita su `WeaponCraft`. Con `OPTIONAL MATCH` buscamos si cada item lo dropea algún monstruo. Si existe monstruo, lo guardamos en la lista de monstruos; si no existe, consideramos ese item como material extra. Por eso usamos el `CASE WHEN m IS NULL THEN i.name END` y luego limpiamos los `null` con comprensión de listas.

El resultado indica que el arma es **`Decapitagallinas I`**, de tipo **`great-sword`**. Para fabricarla hay que matar a **`Yian Kut-Ku`**, y además hace falta el material extra **`Carbalita`**, que no sale de ningún monstruo. 


### Consulta 4:
Por cada tipo de arma, mostrar el nombre del arma o armas de ese tipo que más
daño combinado hace, es decir, la suma de su daño base más todos los daños extra
que sean de kind “element”. Devolver los tipos ordenados de mayor cantidad a
menor cantidad de daño y mostrar el daño que hace dicha arma.


Este va a ser el procedimiento a seguir:
1. Coger todas las armas.
2. Buscae sus daños elementales, si los tienen.
3. Calculae dañoTotal = daño base + suma de daños elementales.
4. Agrupae por tipo de arma.
5. Buscae el daño máximo dentro de cada tipo.
6. Devolver las armas que tienen ese máximo.
7. Ordenar los tipos por ese daño máximo.

In [7]:
query4 = """
MATCH (w:Weapon)
OPTIONAL MATCH (w)-[d:DEALS]->(:Element {kind: 'element'})

WITH w,
     coalesce(w.damage, 0) + coalesce(sum(d.damage), 0) AS dañoTotal

WITH w.kind AS tipo,
     max(dañoTotal) AS maxDaño,
     collect({nombre: w.name, daño: dañoTotal}) AS armas

RETURN tipo,
       [a IN armas WHERE a.daño = maxDaño | a.nombre] AS armas,
       maxDaño AS dañoCombinado

ORDER BY dañoCombinado DESC
"""

df4 = ejecutar_query(
    "4. Arma o armas con mayor daño combinado por tipo",
    query4
)

## 4. Arma o armas con mayor daño combinado por tipo

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (w:Weapon)\nOPTIONAL MATCH (w)-[d:DEALS]->(:Element {kind: 'element'})\n\nWITH w,\n     coalesce(w.damage, 0) + coalesce(sum(d.damage), 0) AS dañoTotal\n\nWITH w.kind AS tipo,\n     max(dañoTotal) AS maxDaño,\n     collect({nombre: w.name, daño: dañoTotal}) AS armas\n\nRETURN tipo,\n       [a IN armas WHERE a.daño = maxDaño | a.nombre] AS armas,\n       maxDaño AS dañoCombinado\n\nORDER BY dañoCombinado DESC\n"


,tipo,armas,dañoCombinado
0,great-sword,[Pez arpón congelante],290
1,gunlance,[Falarmata de las mareas],290
2,hunting-horn,[Cantabile Kut-Ku],280
3,charge-blade,[Valeroje de las mareas],275
4,long-sword,[Khlunda de las mareas],275
5,hammer,[Trueke de Kut-Ku],270
6,switch-axe,[Olacha de las mareas],265
7,bow,[Khviluk de las mareas],260
8,dual-blades,[Ngarpatu de las mareas],260
9,lance,"[Kaminet de fuego, Beaumains purificador G.]",260


sta consulta calcula, para cada tipo de arma, cuál es el arma o armas con mayor daño combinado. El daño combinado se obtiene sumando el daño base de `Weapon` más todos los daños elementales conectados con `DEALS`, pero solo si el elemento tiene `kind = 'element'`. Usamos `OPTIONAL MATCH` para no perder armas que no tengan daño elemental, y `coalesce` para que esos casos cuenten como 0 y no rompan la suma.

Después agrupamos por `w.kind`, calculamos el máximo daño dentro de cada tipo y devolvemos todas las armas que empatan con ese máximo. Esto es mejor que quedarnos solo con una, porque el enunciado dice “arma o armas”.

En los resultados aparecen **14 tipos de arma**. Los mayores daños combinados son `great-sword` y `gunlance`, ambos con **290**. También se ve que la consulta maneja empates, por ejemplo en `lance`, donde aparecen dos armas con daño combinado **260**. Esto confirma que la parte de `[a IN armas WHERE a.daño = maxDaño | a.nombre]` está funcionando correctamente.


### Consulta 5:
El proceso necesario para obtener el arma con id 297. Dicha arma no se puede
craftear directamente, solo se puede obtener mediante actualización de otras armas.
La consulta debe devolver todas las armas intermedias que debo actualizar desde
un arma que sí sea crafteable directamente. Para el arma 297, dicho proceso es [297,
296, 295], es decir, se debe empezar creando el arma 295, luego actualizar el arma
295 a la 296 y por último actualizar la 296 a la 297.


In [8]:
query5 = """
MATCH path = (w2:Weapon {id:297})<-[:TRANSFORMS*]-(w1:Weapon)
WHERE EXISTS { (w1)<-[:PRODUCES]-(:WeaponCraft) }

WITH [n IN nodes(path) WHERE n:Weapon | n.id] AS weapons

RETURN weapons AS proceso
"""

df5 = ejecutar_query(
    "5. Proceso para obtener el arma con id 297",
    query5
)

## 5. Proceso para obtener el arma con id 297

,proceso
0,"[297, 296, 295]"


Como el arma 297 no se puede craftear directamente y necesita al menos una mejora, lo mejor es usar: -[:TRANSFORMS*]->, una o más relaciones.

La consulta busca el camino de actualizaciones necesario para llegar al arma con id `297`. Aquí usamos un `path` con `TRANSFORMS*`, porque no sabemos cuántas mejoras hay entre el arma base y el arma final. La condición `WHERE EXISTS { (w1)<-[:PRODUCES]-(:WeaponCraft) }` asegura que el inicio de la cadena sea un arma que sí se puede fabricar directamente.

Una duda importante era si hacía falta `reverse`. En la versión usada, el patrón está escrito empezando desde el arma final `(w2:Weapon {id:297})` y caminando hacia atrás: `(w2)<-[:TRANSFORMS*]-(w1)`. Por eso `nodes(path)` ya devuelve los nodos en el orden `[297, 296, 295]`, que es justo el orden que pide el enunciado. Si hubiésemos escrito el camino desde la base hacia el target, entonces sí habría hecho falta `reverse`.

El resultado obtenido es exactamente **`[297, 296, 295]`**, así que coincide con el ejemplo del enunciado: se empieza fabricando el arma 295, luego se actualiza a 296 y finalmente a 297.


In [9]:
query5_alt = """
MATCH path = (base:Weapon)-[:TRANSFORMS*]->(target:Weapon {id: 297})
WHERE EXISTS { (:WeaponCraft)-[:PRODUCES]->(base) }

WITH [n IN nodes(path) WHERE n:Weapon | n.id] AS armas

RETURN reverse(armas) AS proceso
"""

df5_alt = ejecutar_query(
    "5. Proceso para obtener el arma con id 297 - versión base a target",
    query5_alt
)

## 5. Proceso para obtener el arma con id 297 - versión base a target

,proceso
0,"[297, 296, 295]"


### Consulta 6:
El nombre de los monstruos que tengo que matar para fabricar el arma con id 880.
Ten en cuenta que dicha arma no se puede craftear directamente, solo se puede
obtener mediante actualización de otras armas.

In [10]:
query6 = """
MATCH path = (target:Weapon {id: 880})<-[:TRANSFORMS*]-(base:Weapon)<-[:PRODUCES]-(wc:WeaponCraft)

UNWIND [n IN nodes(path) WHERE n:WeaponCraft OR n:WeaponUpgrade] AS step

MATCH (step)-[:NEEDS]->(:Item)<-[:DROPS]-(m:Monster)

RETURN DISTINCT m.name AS monstruo
ORDER BY monstruo
"""

df6 = ejecutar_query(
    "6. Monstruos necesarios para fabricar el arma con id 880",
    query6
)

## 6. Monstruos necesarios para fabricar el arma con id 880

,monstruo
0,Gravios
1,Gypceros
2,Jin Dahaad
3,Lala Barina
4,Nerscylla
5,Yian Kut-Ku


In [11]:
query6 = """
MATCH path = (base:Weapon)-[:TRANSFORMS*]->(target:Weapon {id: 880})
WHERE EXISTS { (:WeaponCraft)-[:PRODUCES]->(base) }

WITH path
ORDER BY length(path) ASC
LIMIT 1

WITH [n IN nodes(path) WHERE n:Weapon | n] AS armas

UNWIND armas AS arma

OPTIONAL MATCH (arma)<-[:PRODUCES]-(wc:WeaponCraft)-[:NEEDS]->(:Item)<-[:DROPS]-(m1:Monster)

OPTIONAL MATCH (arma)<-[:TRANSFORMS]-(wu:WeaponUpgrade)-[:NEEDS]->(:Item)<-[:DROPS]-(m2:Monster)

WITH collect(m1.name) + collect(m2.name) AS monstersRaw

UNWIND monstersRaw AS monster

WITH DISTINCT monster
WHERE monster IS NOT NULL

RETURN monster AS monstruo
ORDER BY monstruo
"""

df6 = ejecutar_query(
    "6. Monstruos necesarios para fabricar el arma con id 880",
    query6
)

## 6. Monstruos necesarios para fabricar el arma con id 880

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH path = (base:Weapon)-[:TRANSFORMS*]->(target:Weapon {id: 880})\nWHERE EXISTS { (:WeaponCraft)-[:PRODUCES]->(base) }\n\nWITH path\nORDER BY length(path) ASC\nLIMIT 1\n\nWITH [n IN nodes(path) WHERE n:Weapon | n] AS armas\n\nUNWIND armas AS arma\n\nOPTIONAL MATCH (arma)<-[:PRODUCES]-(wc:WeaponCraft)-[:NEEDS]->(:Item)<-[:DROPS]-(m1:Monster)\n\nOPTIONAL MATCH (arma)<-[:TRANSFORMS]-(wu:WeaponUpgrade)-[:NEEDS]->(:Item)<-[:DROPS]-(m2:Monster)\n\nWITH collect(m1.name) + collect(m2.name) AS monsters

,monstruo
0,Gravios
1,Gypceros
2,Jin Dahaad
3,Lala Barina
4,Nerscylla
5,Yian Kut-Ku


### Comentario de la consulta 6

En esta consulta queremos saber qué monstruos hay que matar para fabricar el arma con id `880`, que no se obtiene directamente, sino mediante una cadena de mejoras. La idea es construir el camino desde un arma base crafteable hasta el arma objetivo y luego mirar todos los pasos de fabricación del camino.

La parte importante es que no miramos los nodos `Weapon` directamente para buscar materiales, sino los nodos que realmente tienen relaciones `NEEDS`: `WeaponCraft` y `WeaponUpgrade`. Por eso usamos `UNWIND [n IN nodes(path) WHERE n:WeaponCraft OR n:WeaponUpgrade] AS step`. Después, para cada paso, buscamos los items necesarios y los monstruos que los dropean.

El resultado devuelve **6 monstruos distintos**: `Gravios`, `Gypceros`, `Jin Dahaad`, `Lala Barina`, `Nerscylla` y `Yian Kut-Ku`. Esto significa que, considerando todo el proceso de creación y mejora hasta el arma 880, esos son los monstruos que aportan materiales necesarios. En el notebook también probamos una versión con camino más corto y salió el mismo resultado, así que en esta base de datos no parece haber conflicto con caminos alternativos para este caso.


### Consulta 7:
El arma con el proceso de fabricación más complejo, es decir, que se necesite pasar
por más armas intermedias para llegar a ella desde un arma que es crafteable
directamente. Debes devolver, el nombre del arma, el tipo del arma, el nombre de
todas las armas por las que debo pasar para llegar a ella y el número de pasos
necesarios.

In [12]:
query7 = """
MATCH path = (base:Weapon)-[:TRANSFORMS*]->(target:Weapon)
WHERE EXISTS { (:WeaponCraft)-[:PRODUCES]->(base) }
  AND NOT EXISTS { (:WeaponCraft)-[:PRODUCES]->(target) }

WITH target,
     [n IN nodes(path) WHERE n:Weapon | n.name] AS armas,
     size([n IN nodes(path) WHERE n:Weapon]) - 1 AS numero_de_pasos

ORDER BY numero_de_pasos DESC
LIMIT 1

RETURN target.name AS arma,
       target.kind AS tipo,
       armas AS armas_intermedias,
       numero_de_pasos
"""

df7 = ejecutar_query(
    "7. Arma con el proceso de fabricación más complejo",
    query7
)

## 7. Arma con el proceso de fabricación más complejo

,arma,tipo,armas_intermedias,numero_de_pasos
0,Bors leal,gunlance,"[Lanza pistola ósea I, Culebrina Quematrice I,...",6


Esta consulta busca el arma con el proceso de fabricación más complejo. Para ello recorremos caminos de `TRANSFORMS*` desde un arma base que sí sea crafteable directamente hasta un arma final que no se pueda craftear directamente. Añadimos `NOT EXISTS` sobre el `target` para centrarnos en armas que realmente dependen de actualizaciones y no en armas que también podrían fabricarse desde cero.

Para contar los pasos usamos `size([n IN nodes(path) WHERE n:Weapon]) - 1`. Esto lo preferimos a `length(path)` porque el camino puede incluir nodos `WeaponUpgrade`, y lo que nos interesa es cuántas transiciones entre armas hay. Si la cadena tiene 7 armas, hay 6 pasos de actualización.

El resultado obtenido es **`Bors leal`**, de tipo **`gunlance`**, con **6 pasos**. La lista de armas intermedias muestra toda la cadena que habría que seguir desde el arma base hasta llegar a ella. Esta versión compara todos los caminos encontrados; si hubiera muchos caminos alternativos por target y quisiéramos el “mínimo proceso necesario” por arma, se podría usar un `CALL` para elegir primero el camino mínimo de cada target. 

### Consulta 8:
El arma con el proceso de fabricación que requiere matar más monstruos diferentes,
puedes obviar los empates. Debes devolver el nombre del arma, el nombre de todas
las armas por las que debo pasar para llegar a ella y el número de monstruos que es necesario matar.


In [13]:
query8 = """
MATCH path = (wc:WeaponCraft)-[:PRODUCES]->(base:Weapon)-[:TRANSFORMS*0..]->(target:Weapon)

UNWIND [n IN nodes(path) WHERE n:WeaponCraft OR n:WeaponUpgrade] AS step

MATCH (step)-[:NEEDS]->(:Item)<-[:DROPS]-(m:Monster)

WITH target.name AS nombre,
     [n IN nodes(path) WHERE n:Weapon | n.name] AS armas,
     collect(DISTINCT m.name) AS monstruos

RETURN nombre,
       armas,
       monstruos,
       size(monstruos) AS numero_monstruos

ORDER BY numero_monstruos DESC
LIMIT 1
"""

df8 = ejecutar_query(
    "8. Arma cuyo proceso requiere matar más monstruos diferentes",
    query8
)

## 8. Arma cuyo proceso requiere matar más monstruos diferentes

,nombre,armas,monstruos,numero_monstruos
0,Arco recio de cazador,"[Arco de cazador I, Arco de cazador II, Arco d...","[Lala Barina, Jin Dahaad, Chatacabra, Balahara...",13


Esta consulta es parecida a la anterior, pero ahora no buscamos el proceso con más pasos, sino el proceso que obliga a matar más monstruos diferentes. Aquí usamos `TRANSFORMS*0..` porque el arma final puede ser directamente el arma base; es decir, también contemplamos armas que se fabrican directamente sin upgrades.

La consulta mete el `WeaponCraft` dentro del camino para no perder los materiales de la creación inicial. Después hace `UNWIND` de los pasos que pueden necesitar materiales (`WeaponCraft` y `WeaponUpgrade`) y busca qué monstruos dropean esos items. Usamos `collect(DISTINCT m.name)` porque el enunciado pide monstruos diferentes; si el mismo monstruo dropea varios materiales, solo debe contarse una vez.

El resultado indica que el arma es **`Arco recio de cazador`**, con una cadena de armas que empieza en `Arco de cazador I` y llega hasta el arma final. El proceso requiere matar **13 monstruos diferentes**, así que es el proceso más exigente en variedad de monstruos. Como el enunciado permite obviar empates, usamos `ORDER BY numero_monstruos DESC LIMIT 1`.


### Consulta 9:
Escribe una consulta que devuelva para cada ubicación (Location), el o los
elementos (Element) de tipo "element" frente a los que los monstruos que viven allí
son más débiles en conjunto (sumando los niveles de debilidad), y devuelve el
nombre de la ubicación, el valor total de esa máxima debilidad y los nombres de
dichos elementos.

In [14]:
query9 = """
MATCH (l:Location)<-[:LIVES_IN]-(m:Monster)-[w:WEAK_TO]->(e:Element {kind: 'element'})

WITH l,
     e.name AS elemento,
     sum(w.level) AS debilidadTotal

WITH l,
     collect({elemento: elemento, debilidad: debilidadTotal}) AS debilidades,
     max(debilidadTotal) AS maxDebilidad

RETURN l.name AS ubicacion,
       maxDebilidad AS debilidadMaxima,
       [d IN debilidades WHERE d.debilidad = maxDebilidad | d.elemento] AS elementos

ORDER BY ubicacion
"""

df9 = ejecutar_query(
    "9. Elementos con mayor debilidad acumulada por ubicación",
    query9
)

## 9. Elementos con mayor debilidad acumulada por ubicación

,ubicacion,debilidadMaxima,elementos
0,Acantilados del Témpano,6,[fire]
1,Bosque Escarlata,3,"[fire, dragon]"
2,Cuenca Oleosa,4,"[water, dragon]"
3,Llanos Barlovento,2,"[fire, thunder, dragon]"
4,Ruinas de Wyveria,8,[fire]


Esta consulta trabaja por localización. Primero buscamos los monstruos que viven en cada `Location` y sus debilidades elementales mediante `WEAK_TO`. Después agrupamos por ubicación y elemento, sumando los niveles de debilidad. Esto nos da, por ejemplo, cuánto de vulnerable es en conjunto una localización frente a `fire`, `water`, `dragon`, etc.

Luego agrupamos de nuevo por ubicación y calculamos la máxima debilidad acumulada. La lista final de elementos se obtiene filtrando los que tienen esa debilidad máxima. Esto es importante porque el enunciado pide “el o los elementos”, así que si hay empate se devuelven varios.

Los resultados muestran que `Ruinas de Wyveria` tiene una debilidad máxima de **8** frente a `fire`, y `Acantilados del Témpano` tiene **6** también frente a `fire`. En cambio, hay localizaciones con empates: `Bosque Escarlata` es más vulnerable a `[fire, dragon]`, `Cuenca Oleosa` a `[water, dragon]` y `Llanos Barlovento` a `[fire, thunder, dragon]`. Esto confirma que la consulta está gestionando bien los empates.


### Consulta 10:
Escribe una consulta que, dado un tipo de arma como parámetro, devuelva qué arma
deberías llevar a cada localización. Para ello, se seleccionarán las armas del tipo indicado que causen más daño contra los elementos frente a los que, en conjunto,
los monstruos de esa localización son más débiles (sumando sus niveles de
debilidad). La consulta debe devolver el nombre de la ubicación, los elementos
frente a los que los monstruos son más vulnerables y el nombre de las armas
recomendadas.

In [15]:
query_tipos = """
MATCH (w:Weapon)
RETURN DISTINCT w.kind AS tipo
ORDER BY tipo
"""

df_tipos = ejecutar_query("Tipos de arma disponibles", query_tipos)

## Tipos de arma disponibles

,tipo
0,bow
1,charge-blade
2,dual-blades
3,great-sword
4,gunlance
5,hammer
6,heavy-bowgun
7,hunting-horn
8,insect-glaive
9,lance


In [16]:
query10 = """
MATCH (l:Location)<-[:LIVES_IN]-(m:Monster)-[weak:WEAK_TO]->(e:Element {kind: 'element'})

WITH l,
     e,
     sum(weak.level) AS debilidadTotal

WITH l,
     collect({
       elemento: e,
       nombre: e.name,
       debilidad: debilidadTotal
     }) AS debilidades,
     max(debilidadTotal) AS maxDebilidad

WITH l,
     [d IN debilidades WHERE d.debilidad = maxDebilidad | d.elemento] AS elementosVulnerables,
     [d IN debilidades WHERE d.debilidad = maxDebilidad | d.nombre] AS nombresElementos

MATCH (arma:Weapon {kind: $tipoArma})
OPTIONAL MATCH (arma)-[deal:DEALS]->(e:Element {kind: 'element'})

WITH l,
     nombresElementos,
     arma,
     sum(
       CASE
         WHEN e IN elementosVulnerables THEN coalesce(deal.damage, 0)
         ELSE 0
       END
     ) AS dañoContraElementos

WITH l,
     nombresElementos,
     collect({arma: arma.name, daño: dañoContraElementos}) AS armas,
     max(dañoContraElementos) AS maxDaño

RETURN l.name AS ubicacion,
       nombresElementos AS elementosVulnerables,
       [a IN armas WHERE a.daño = maxDaño | a.arma] AS armasRecomendadas

ORDER BY ubicacion
"""

df10 = ejecutar_query(
    "10. Arma recomendada por ubicación según tipo de arma",
    query10,
    params={"tipoArma": "great-sword"}
)

## 10. Arma recomendada por ubicación según tipo de arma

,ubicacion,elementosVulnerables,armasRecomendadas
0,Acantilados del Témpano,[fire],[Decapitagallos]
1,Bosque Escarlata,"[fire, dragon]",[Decapitagallos]
2,Cuenca Oleosa,"[water, dragon]",[Lamorak reforzado]
3,Llanos Barlovento,"[fire, thunder, dragon]",[Decapitagallos]
4,Ruinas de Wyveria,[fire],[Decapitagallos]


Esta consulta es una recomendación de arma por localización dado un tipo de arma como parámetro. Aquí se probó con `tipoArma = 'great-sword'`. Primero reutilizamos la lógica de la consulta 9 para saber qué elementos son los más efectivos contra los monstruos de cada ubicación. Después buscamos todas las armas del tipo indicado y calculamos cuánto daño hacen contra esos elementos vulnerables.

La parte del `CASE` es la clave: solo suma el daño elemental de un arma si el elemento de ese daño está dentro de los elementos más vulnerables de la ubicación. Si no coincide, suma 0. Luego agrupamos las armas y nos quedamos con las que tengan el máximo daño contra esos elementos. Como devolvemos una lista, también se podrían recoger empates.

El resultado para `great-sword` recomienda `Decapitagallos` en casi todas las ubicaciones, porque funciona bien contra `fire`, que aparece como elemento vulnerable en varias zonas. Para `Cuenca Oleosa`, donde los elementos más vulnerables son `[water, dragon]`, recomienda **`Lamorak reforzado`**. Esto tiene sentido: la recomendación cambia cuando cambian los elementos débiles de la zona.
